# Morphing Fitness Landscape

Interactive 3D visualisation of a combinatorial protein fitness landscape
morphing from additive (smooth) to epistatic (rugged) as alpha increases.

**Protein**: Green Fluorescent Protein (GFP) from *Aequorea victoria*
- PDB: [2WUR](https://www.rcsb.org/structure/2WUR) (superfolder GFP, chain A)
- DMS: 1,084 single-site fluorescence measurements across 233 positions (Sarkisyan et al., 2016)
- 238 residues, 1,114 structural contacts at 8A threshold

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from Bio import SeqIO

from mutascape import (
    ContactMap,
    Landscape,
    RandomEpistasis,
    SignConsistentEpistasis,
)

## Build the landscape

Load GFP's real DMS data (Sarkisyan et al., 2016) and structural contact map
from PDB 2WUR (8A C-beta threshold). Pairwise epistatic couplings are generated
synthetically using `RandomEpistasis`, constrained to structurally contacting
residue pairs.

In [2]:
DATA_DIR = "../data/gfp"

wt_seq = str(next(SeqIO.parse(f"{DATA_DIR}/gfp_wt.fasta", "fasta")).seq)

# Load pre-extracted distance matrix (avoids re-parsing the PDB)
data = np.load(f"{DATA_DIR}/gfp_distances.npz")
contacts = ContactMap.from_distance_matrix(
    data["distances"], data["positions"].tolist(), threshold=8.0
)

landscape = Landscape.from_dms(
    dms_path=f"{DATA_DIR}/gfp.csv",
    contact_map=contacts,
    epistasis=RandomEpistasis(sigma=0.3, seed=42),
    alpha=1.0,
    seed=42,
    wt_sequence=wt_seq,
)

print(landscape)
print(f"  {contacts.n_contacts} structural contacts across {contacts.n_positions} positions")

Landscape(positions=[3(K), 4(G), 5(E), 6(E), 7(L), 8(F), 9(T), 10(G), 11(V), 12(V), 13(P), 14(I), 15(L), 16(V), 17(E), 18(L), 19(D), 20(G), 21(D), 22(V), 23(N), 24(G), 25(H), 26(K), 27(F), 28(S), 29(V), 30(S), 31(G), 32(E), 33(G), 34(E), 35(G), 36(D), 37(A), 38(T), 39(Y), 40(G), 41(K), 42(L), 43(T), 44(L), 45(K), 46(F), 47(I), 48(C), 49(T), 50(T), 51(G), 52(K), 53(L), 54(P), 55(V), 56(P), 57(W), 58(P), 59(T), 60(L), 61(V), 62(T), 63(T), 64(L), 65(S), 66(Y), 67(G), 68(V), 69(Q), 70(C), 71(F), 72(S), 73(R), 74(Y), 75(P), 76(D), 77(H), 78(M), 79(K), 80(Q), 81(H), 82(D), 83(F), 84(F), 85(K), 86(S), 87(A), 88(M), 89(P), 90(E), 91(G), 92(Y), 93(V), 94(Q), 95(E), 96(R), 97(T), 98(I), 99(F), 100(F), 101(K), 102(D), 103(D), 104(G), 105(N), 106(Y), 107(K), 108(T), 109(R), 110(A), 111(E), 112(V), 113(K), 114(F), 115(E), 116(G), 117(D), 118(T), 121(N), 122(R), 123(I), 124(E), 125(L), 126(K), 127(G), 128(I), 129(D), 130(F), 131(K), 132(E), 133(D), 134(G), 135(N), 136(I), 137(L), 138(G), 139(H), 140

/Users/tomhill/Uni/research-project/mutascape/src/mutascape/landscape.py:663: UserWarning: 9 positions not found in contact map and will have no epistatic interactions.
  warnings.warn(


## Morphing animation

The key parameter is **alpha** (epistatic strength multiplier):

| alpha | Behaviour |
|-------|-----------|
| 0.0 | Purely additive. Multi-mutant fitness = sum of single-site effects. Smooth surface. |
| 0.5 | Mild epistasis. Surface begins to buckle at structurally contacting sites. |
| 1.0 | Moderate epistasis. Local optima start appearing. |
| 2.0+ | Strong epistasis. Rugged landscape with sign epistasis and fitness traps. |

The animation samples 3,000 combinatorial variants (1 to 4 mutations each),
embeds their mutation profiles into 2D via PCA, then sweeps alpha while
keeping the same set of variants fixed in x/y. The z-axis (fitness) morphs
as epistasis reshapes the landscape.

Use the **Play** button or drag the slider to sweep alpha.

In [3]:
fig = landscape.plot_morphing(
    n_samples=3000,
    n_mutations=(1, 4),
    alphas=[0.0, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0],
    embed_method="pca",
    colorscale="Viridis",
    grid_resolution=60,
    seed=42,
)
fig.show()

## Export as standalone HTML

The figure can be saved as a self-contained HTML file for sharing or embedding in presentations.

In [4]:
fig.write_html("gfp_morphing_landscape.html", include_plotlyjs=True)
print("Saved to gfp_morphing_landscape.html")

Saved to gfp_morphing_landscape.html


## Comparing epistasis models

`SignConsistentEpistasis` produces smoother ridges where beneficial mutations
cooperate. `RandomEpistasis` is more chaotic. Compare how the surfaces differ
at the same alpha.

In [5]:
landscape_sc = Landscape.from_dms(
    dms_path=f"{DATA_DIR}/gfp.csv",
    contact_map=contacts,
    epistasis=SignConsistentEpistasis(sigma=0.3, seed=42),
    alpha=1.0,
    seed=42,
    wt_sequence=wt_seq,
)

fig_sc = landscape_sc.plot_morphing(
    n_samples=3000,
    n_mutations=(1, 4),
    alphas=[0.0, 0.5, 1.0, 2.0, 5.0],
    colorscale="Plasma",
    title="GFP: SignConsistentEpistasis",
    seed=42,
)
fig_sc.show()

/Users/tomhill/Uni/research-project/mutascape/src/mutascape/landscape.py:663: UserWarning: 9 positions not found in contact map and will have no epistatic interactions.
  warnings.warn(


## Quantifying the morphing

The `stats()` method measures how far the landscape deviates from additivity.
Compare across the same alpha values used in the animation.

In [6]:
import pandas as pd

rows = []
for alpha in [0.0, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0]:
    ls = Landscape(
        wild_type=landscape.wild_type,
        single_site_effects=landscape.single_site_effects,
        contact_map=landscape.contact_map,
        couplings=landscape.couplings,
        alpha=alpha,
        seed=42,
        wt_sequence=wt_seq,
    )
    s = ls.stats(n_samples=2000, n_mutations=3)
    rows.append({"alpha": alpha, **s})

stats_df = pd.DataFrame(rows).set_index("alpha")
stats_df.style.format("{:.3f}").background_gradient(axis=0, cmap="RdYlGn_r")

,mean_fitness,std_fitness,additive_correlation,mean_epistatic_deviation,fraction_sign_epistasis
alpha,,,,,
0.000000,-1.152,1.269,1.000,0.000,0.000
0.250000,-1.153,1.268,0.999,0.011,0.038
0.500000,-1.155,1.269,0.998,0.023,0.052
1.000000,-1.157,1.275,0.992,0.046,0.061
2.000000,-1.161,1.301,0.967,0.092,0.067
3.000000,-1.166,1.348,0.930,0.138,0.071
5.000000,-1.174,1.493,0.833,0.230,0.074
